In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
import joblib

# Cell 2: Load Data
df = pd.read_csv('EURUSD_clean.csv', index_col=0, parse_dates=True)
print(f"Original shape: {df.shape}")
print(df.head())

# Cell 3: Handle Missing Values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Forward fill then backward fill for any remaining
df = df.fillna(method='ffill').fillna(method='bfill')
print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Cell 4: Remove Outliers using IQR method
def remove_outliers_iqr(df, column, multiplier=3):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_clean, outliers

for col in ['Open', 'High', 'Low', 'Close']:
    df, removed = remove_outliers_iqr(df, col, multiplier=3)
    print(f"Removed {len(removed)} outliers from {col}")

print(f"\nShape after outlier removal: {df.shape}")

# Cell 5: Normalization Options
# Choose scaler based on data distribution
scaler = RobustScaler()  # Better for financial data with outliers

# Scale price columns
price_cols = ['Open', 'High', 'Low', 'Close']
df_scaled = df.copy()
df_scaled[price_cols] = scaler.fit_transform(df[price_cols])

# Volume scaling (log transform first, then scale)
df_scaled['Volume_log'] = np.log1p(df['Volume'])
volume_scaler = RobustScaler()
df_scaled['Volume_scaled'] = volume_scaler.fit_transform(df_scaled[['Volume_log']])
df_scaled = df_scaled.drop(['Volume', 'Volume_log'], axis=1)

print("Data after scaling:")
print(df_scaled.head())
print(f"\nScaled stats - Close price: min={df_scaled['Close'].min():.3f}, max={df_scaled['Close'].max():.3f}")

# Cell 6: Create Sequences for Analysis
def create_analysis_sequences(data, sequence_length=60, step=1):
    """
    Create sequences for time series analysis (not for prediction)
    Returns sequences and corresponding timestamps
    """
    sequences = []
    timestamps = []
    
    for i in range(0, len(data) - sequence_length + 1, step):
        seq = data.iloc[i:i+sequence_length]
        sequences.append(seq)
        timestamps.append(data.index[i+sequence_length-1])
    
    return sequences, timestamps

SEQ_LENGTH = 60
sequences, timestamps = create_analysis_sequences(df_scaled, SEQ_LENGTH)

print(f"Created {len(sequences)} sequences")
print(f"Sequence shape: {sequences[0].shape}")
print(f"Sample timestamp: {timestamps[0]}")

# Cell 7: Split into Train/Validation/Test
# Time-based split (70/15/15)
train_size = int(0.7 * len(sequences))
val_size = int(0.15 * len(sequences))

train_sequences = sequences[:train_size]
val_sequences = sequences[train_size:train_size+val_size]
test_sequences = sequences[train_size+val_size:]

train_timestamps = timestamps[:train_size]
val_timestamps = timestamps[train_size:train_size+val_size]
test_timestamps = timestamps[train_size+val_size:]

print(f"Train: {len(train_sequences)} sequences")
print(f"Validation: {len(val_sequences)} sequences")
print(f"Test: {len(test_sequences)} sequences")

# Cell 8: Convert to DataFrames for saving
def sequences_to_dataframe(sequences, timestamps):
    data_list = []
    for seq, ts in zip(sequences, timestamps):
        seq_df = seq.copy()
        seq_df['sequence_id'] = ts
        seq_df['position_in_sequence'] = range(len(seq_df))
        data_list.append(seq_df)
    
    return pd.concat(data_list, ignore_index=True)

train_df = sequences_to_dataframe(train_sequences, train_timestamps)
val_df = sequences_to_dataframe(val_sequences, val_timestamps)
test_df = sequences_to_dataframe(test_sequences, test_timestamps)

# Cell 9: Save Processed Data
train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)

# Save scalers
joblib.dump(scaler, 'price_scaler.pkl')
joblib.dump(volume_scaler, 'volume_scaler.pkl')

print("Preprocessed data saved successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")